# Predikcija broja biciklista primenom metoda dubokog učenja
**Bulevar oslobođenja, Novi Sad (2021-2024)**

Ovaj kod implementira i evaluira modele rekurentnih neuronskih mreža (Bidirectional LSTM i GRU) za predikciju broja biciklista na satnom i dnevnom nivou.

## 1. Ucitavanje biblioteka i podataka
U prvom koraku, uvoze se potrebne biblioteke i ucitavaju se meteoroloski i saobracajni podaci podeljeni po godinama. Podaci se spajaju u jedinstveni DataFrame.

In [13]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,Dense,Dropout,GRU
from sklearn.metrics import mean_absolute_percentage_error,mean_absolute_error,mean_squared_error
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Bidirectional, BatchNormalization


# Učitavanje i spajanje podataka
files = ['meteo_podaci_2021.xlsx', 'meteo_podaci_2022.xlsx', 'meteo_podaci_2023.xlsx', 'meteo_podaci_2024.xlsx']
df_list = []

for f in files:
    temp_df = pd.read_excel(f)
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)
print(f"Loaded {len(df)} rows of data.")

Loaded 35064 rows of data.


## 2. Pretprocesiranje podataka
Predikcija vremenskih serija zahteva da model razume kontekst. U ovom koraku radi se sledece:
1. Formira se tacan vremenski indeks (`Datetime`) iz kolona Datum i Cas.
2. Uvodi se **autoregresivne karakteristike (Lagove)**. Modelu se daje informaciju koliko je bicikala prošlo pre tacno 24 sata i pre tacno 7 dana, sto mu pomaze da nauci dnevne i nedeljne cikluse.

In [14]:
# 1. Kreiranje timestamp indeksa
sati = df["Cas"].astype(str).str.split(':').str[0].astype(int)
df['Datetime'] = pd.to_datetime(df["Datum"] + pd.to_timedelta(sati, unit='h'))
df.set_index('Datetime', inplace=True)
df.sort_index(inplace=True)

# 2. Dodavanje autoregresivnih lagova
df['lag_24h'] = df['br_bicikala'].shift(24)      # Broj bicikala pre tačno jedan dan
df['lag_7dana'] = df['br_bicikala'].shift(168)   # Broj bicikala u isto vreme pre tačno nedelju dana

# Brisanje redova koji su zbog lagovanja dobili NaN (prazne) vrednosti
df.dropna(inplace=True) 

# Pretvaranje kategoričke varijable 'godisnje_doba' u numeričke dummy kolone
df = pd.get_dummies(df, columns=["godisnje_doba"], drop_first=True)

# Izbacivanje nepotrebnih kolona
delete_columns = ['Datum', 'Cas']
X = df.drop(columns=delete_columns)
y = df['br_bicikala']


## 3. Priprema podataka

* **Podela skupa:** Prve tri godine (2021-2023) koriste se za trening, dok se 2024. godina zadrzava iskljucivo za testiranje.
* **Skaliranje:** Vrednosti se svode na opseg [0, 1] pomoću `MinMaxScaler`-a.
* **Sekvenciranje:** Primenjuje se tehnika klizeceg prozora (*sliding window*). Model će posmatrati kolonu od prethodnih 24 sata kako bi predvideo desavanja u narednom satu.

In [15]:
# Podela podataka po godinama
train_mask = df['godina'] < 2024
test_mask = df['godina'] == 2024

X_train_raw, y_train_raw = X[train_mask], y[train_mask]
X_test_raw, y_test_raw = X[test_mask], y[test_mask]

# Skaliranje
scalerX = MinMaxScaler()
scalery = MinMaxScaler()

X_train_scaled = scalerX.fit_transform(X_train_raw)
y_train_scaled = scalery.fit_transform(y_train_raw.values.reshape(-1, 1))

X_test_scaled = scalerX.transform(X_test_raw)
y_test_scaled = scalery.transform(y_test_raw.values.reshape(-1, 1))

# Funkcija za kreiranje 3D sekvenci (uzorci, vremenski_koraci, karakteristike)
def create_sequences(X_data, y_data, time_steps):
    X_seq, y_seq = [], []
    for i in range(len(X_data) - time_steps):
        X_seq.append(X_data[i:(i+time_steps)])
        y_seq.append(y_data[i+time_steps])
    return np.array(X_seq), np.array(y_seq)

time_steps = 24
X_train, y_train = create_sequences(X_train_scaled, y_train_scaled, time_steps)
X_test, y_test = create_sequences(X_test_scaled, y_test_scaled, time_steps)

print(f"Shape X_train tensor for training: {X_train.shape}")

Shape X_train tensor for training: (26088, 24, 18)


## 4. Definisanje arhitekture neuronske mreže
Funkcija `build_model` omogucava laku promenu izmedju LSTM i GRU arhitekture. 
Koristi se napredna **Bidirectional** (dvosmernu) struktura koja analizira sekvencu od 24 sata i unapred i unazad, što poboljsava stabilnost. Optimizator je podešen da direktno minimizuje srednju apsolutnu gresku (**MAE**).

In [16]:
def build_model(model_type='LSTM'):
    model = Sequential()

    if model_type == 'LSTM':
        model.add(Bidirectional(LSTM(128, activation='relu', return_sequences=True), input_shape=(X_train.shape[1], X_train.shape[2])))
        model.add(BatchNormalization())
        model.add(Dropout(0.2))
        model.add(Bidirectional(LSTM(64, activation='relu', return_sequences=False)))
        model.add(BatchNormalization())
    elif model_type == 'GRU':
        model.add(Bidirectional(GRU(128, activation='relu', return_sequences=True), input_shape=(X_train.shape[1], X_train.shape[2])))
        model.add(BatchNormalization())
        model.add(Dropout(0.2))
        model.add(Bidirectional(GRU(64, activation='relu', return_sequences=False)))
        model.add(BatchNormalization())
    
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1)) # Izlaz za regresiju

    opt = Adam(learning_rate=0.0005)
    model.compile(optimizer=opt, loss='mae', metrics=['mae'])
    return model

# Možete promeniti 'LSTM' u 'GRU' za testiranje drugog modela
model = build_model(model_type='LSTM') 
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bidirectional (Bidirection  (None, 24, 256)           150528    
 al)                                                             
                                                                 
 batch_normalization (Batch  (None, 24, 256)           1024      
 Normalization)                                                  
                                                                 
 dropout (Dropout)           (None, 24, 256)           0         
                                                                 
 bidirectional_1 (Bidirecti  (None, 128)               164352    
 onal)                                                           
                                                                 
 batch_normalization_1 (Bat  (None, 128)               512       
 chNormalization)                                     

## 5. Treniranje Modela
Pokrece trening uz mehanizam `EarlyStopping` sa granicom od 10 epoha. Ovo sprecava pretreniranje (overfitting) i prekida ucenje onda kada se validaciona greska više ne smanjuje.

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1, 
    callbacks=[early_stop],
    verbose=1
)

# Vizuelizacija opadanja greške tokom treninga
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Trening Loss (MAE)')
plt.plot(history.history['val_loss'], label='Validaciona Loss (MAE)')
plt.title('Istorija učenja modela')
plt.xlabel('Epoha')
plt.ylabel('Loss (skaliran)')
plt.legend()
plt.show()

Epoch 1/50
734/734 [==============================] - 102s 111ms/step - loss: 0.2633 - mae: 0.2633 - val_loss: 0.1083 - val_mae: 0.1083
Epoch 2/50
734/734 [==============================] - 79s 108ms/step - loss: 0.1185 - mae: 0.1185 - val_loss: 0.0708 - val_mae: 0.0708
Epoch 3/50
734/734 [==============================] - 73s 99ms/step - loss: 0.0860 - mae: 0.0860 - val_loss: 0.0626 - val_mae: 0.0626
Epoch 4/50
734/734 [==============================] - 80s 109ms/step - loss: 0.0696 - mae: 0.0696 - val_loss: 0.0570 - val_mae: 0.0570
Epoch 5/50
734/734 [==============================] - 75s 102ms/step - loss: 0.0605 - mae: 0.0605 - val_loss: 0.0527 - val_mae: 0.0527
Epoch 6/50
734/734 [==============================] - 75s 102ms/step - loss: 0.0543 - mae: 0.0543 - val_loss: 0.0448 - val_mae: 0.0448
Epoch 7/50
734/734 [==============================] - 74s 101ms/step - loss: 0.0515 - mae: 0.0515 - val_loss: 0.0437 - val_mae: 0.0437
Epoch 8/50
734/734 [==============================] - 8

## 6. Evaluacija i Vizuelizacija na Satnom Nivou
Testira se model na nevidjenoj 2024. godini. Generisu se predikcije, vracaju se u originalnu skalu (stvarni broj biciklista) i vizuelno se porede sa stvarnim podacima za odredjeni odsecak vremena.

In [ ]:
y_pred_scaled = model.predict(X_test)

y_pred = scalery.inverse_transform(y_pred_scaled)
y_test_real = scalery.inverse_transform(y_test)

rmse = np.sqrt(mean_squared_error(y_test_real, y_pred))
mae = mean_absolute_error(y_test_real, y_pred)

print(f"--- RESULTS PER HOUR (Test set - 2024) ---")
print(f"Hourly RMSE: {rmse:.2f} bicycles")
print(f"Hourly MAE: {mae:.2f} bicycles")

# Vizuelizacija (prikazujemo 200 sati)
show_from = 1000 
show_to = 1200

plt.figure(figsize=(15, 6))
plt.plot(y_test_real[show_from:show_to], label='Real bycicle number', color='blue', alpha=0.7)
plt.plot(y_pred[show_from:show_to], label='Prediction of model', color='red', alpha=0.8, linestyle='--')
plt.title('View: Real vs Predicted number bicycles (Per hour)')
plt.xlabel('Hours')
plt.ylabel('Number bicycles')
plt.legend()
plt.grid(True)
plt.show()

## 7. Evaluacija i Vizuelizacija na Dnevnom Nivou
Kako je cilj projekta predikcija ocekivanog *dnevnog* broja biciklista, u ovom koraku pregrupisu se satne predikcije i stvarna brojanja na nivou svakog datuma, a zatim se racunaju finalne projektne metrike.

In [ ]:
# Ekstrakcija datuma iz indeksa
dates_test = y_test_raw.index[time_steps:]

# Pakovanje u DataFrame za lakše sumiranje
results_df = pd.DataFrame({
    'Real_hour': y_test_real.flatten(),
    'Predicted_hour': y_pred.flatten()
}, index=dates_test)

# Resamplovanje na dnevni nivo ('D' = dan)
daily_results = results_df.resample('D').sum()

# Računanje dnevnih metrika
daily_rmse = np.sqrt(mean_squared_error(daily_results['Real_hour'], daily_results['Predicted_hour']))
daily_mae = mean_absolute_error(daily_results['Real_hour'], daily_results['Predicted_hour'])

print(f"\n--- Finaly daily results ---")
print(f"Daily RMSE: {daily_rmse:.2f} bicycles")
print(f"Daily MAE: {daily_mae:.2f} bicylces")

# Vizuelizacija dnevnih dešavanja u 2024. godini
plt.figure(figsize=(12, 5))
plt.plot(daily_results.index, daily_results['Real_hour'], label='Stvarni dnevni broj', color='blue', alpha=0.7)
plt.plot(daily_results.index, daily_results['Predicted_hour'], label='Predviđeni dnevni broj', color='red', linestyle='--', alpha=0.8)
plt.title('Daily bicycle number on street Bulevar oslobođenja (2024. year)')
plt.xlabel('Date')
plt.ylabel('Total bicycle number per day')
plt.legend()
plt.grid(True)
plt.show()